# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zuhairsyed123/ML_internship_Assignments/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research Question :

Which content pages should be prioritized for refresh, optimization, or monitoring, based on observable search and engagement signals?

Decision it supports :

This ranking supports a single, concrete decision: which pages a content or SEO team reviews first when review time is limited. A content manager acts on the output directly — pulling the top-ranked pages into this week's refresh queue instead of reviewing the catalog in no particular order. Getting it wrong has a real cost either way: a page ranked too high wastes review time on content that didn't need it, while a page ranked too low (or missed) keeps losing traffic unnoticed until the next review cycle.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Download Dataset

First, let's download the `content_refresh_anonymized.csv` dataset from the project's GitHub repository into the Colab environment.

In [3]:
import requests

# URL of the raw CSV file on GitHub
url = "https://raw.githubusercontent.com/Zuhairsyed123/ML_internship_Assignments/main/data/raw/content_refresh_anonymized.csv"

# Local path to save the file in Colab
local_filename = "/content/content_refresh_anonymized.csv"

# Download the file
response = requests.get(url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(local_filename, 'wb') as f:
    f.write(response.content)

print(f"Dataset downloaded to {local_filename}")

Dataset downloaded to /content/content_refresh_anonymized.csv


Now that the dataset is downloaded, you can rerun the cell below (cell `5YsVquMxQrPI`) to load the data into a pandas DataFrame.

In [4]:
import pandas as pd

# --- Release used ---
# Starter release: content_refresh_anonymized.csv (FlyRank ML Internship)
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# --- Date window ---
# Warehouse cross-check used table fact_content_daily_performance,
# month = 2026-03, filtered to pages with >= 100 monthly impressions.

# --- Exclusions ---
# Warehouse rows below the 100-impression floor were dropped to avoid
# scoring pages with too little traffic to measure CTR or position reliably.

print("Starter release: content_refresh_anonymized.csv")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print()
print("Warehouse cross-check: fact_content_daily_performance, month=2026-03")
print("Rows after >=100 impressions filter: 101,441")

Starter release: content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Warehouse cross-check: fact_content_daily_performance, month=2026-03
Rows after >=100 impressions filter: 101,441


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [8]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

# --- Label definition ---
# Proxy label: page is "declining" if its logged trend direction is down.
# This is a defined proxy, not a directly measured traffic drop.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# --- Features ---
# Log-transform skewed count features (raw impressions/clicks are heavily
# right-skewed; log1p brings them closer to a usable scale for the models).
for col in ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d']:
    df[f'log_{col}'] = np.log1p(df[col].fillna(0))

# Striking-distance flag: pages ranking 4-10 are close enough to page 1
# that a small CTR improvement is plausible.
df['is_striking'] = ((df['avg_position'] >= 4) & (df['avg_position'] <= 10)).astype(int)
# --- Fill missing values (same logic as w05) ---
fill_cols = {
    'avg_position': 0,
    'ctr': 0,
    'engagement_rate': 0,
    'scroll_rate': 0,
    'word_count': df['word_count'].median(),
    'search_volume': 0,
    'competition': 0,
    'days_since_last_update': df['days_since_last_update'].median()
}
for col, val in fill_cols.items():
    df[f'{col}_filled'] = df[col].fillna(val)

feature_cols = [
    'log_impressions_90d', 'log_clicks_90d', 'log_pageviews_90d', 'log_sessions_90d',
    'avg_position_filled', 'ctr_filled', 'engagement_rate_filled', 'scroll_rate_filled',
    'word_count_filled', 'search_volume_filled', 'competition_filled',
    'days_since_last_update_filled', 'is_striking'
]

# --- Baseline ---
# Rule-based baseline: only flags striking-distance pages, scored by a
# 50/50 blend of impression volume and inverse CTR percentile.
df['imp_rank'] = df['impressions_90d'].rank(pct=True)
df['ctr_rank_desc'] = 1.0 - df['ctr'].rank(pct=True)
df['baseline_score'] = df['is_striking'] * (0.5 * df['imp_rank'] + 0.5 * df['ctr_rank_desc'])

# --- Validation design ---
# Grouped split by client_id, NOT a random row split — this stops a model
# from just memorizing per-client baselines instead of a general pattern.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df['is_declining_label'], df['client_id']))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

# --- Leakage check ---
# A deliberately "leaked" version of this model, trained WITH imp_future
# (the same future window used to build the label), hit ~97% accuracy —
# the giveaway that a feature was contaminated. Removing imp_future and
# re-testing under the grouped split above is what keeps this model honest.
assert 'imp_future' not in feature_cols, "Leakage: imp_future must never enter the feature set"

print("Train pages:", len(train_df), "| Test pages:", len(test_df))
print("Declining rate — train:", round(train_df['is_declining_label'].mean(), 4),
      "| test:", round(test_df['is_declining_label'].mean(), 4))

Train pages: 23837 | Test pages: 6163
Declining rate — train: 0.5501 | test: 0.511


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

X_train, y_train = train_df[feature_cols], train_df['is_declining_label']
X_test, y_test = test_df[feature_cols], test_df['is_declining_label']

def precision_at_k(scores, labels, k):
    top_idx = np.argsort(scores)[-k:]
    return labels.iloc[top_idx].mean()

# --- Baseline scores (already computed on test_df in section 3) ---
baseline_scores = test_df['baseline_score']

# --- Models, all trained on the identical split ---
models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
}

results = []
results.append({
    "model": "Baseline (rule-based)",
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
    "precision_at_50": precision_at_k(baseline_scores, y_test, 50),
    "precision_at_100": precision_at_k(baseline_scores, y_test, 100),
})

for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, probs),
        "avg_precision": average_precision_score(y_test, probs),
        "precision_at_50": precision_at_k(probs, y_test, 50),
        "precision_at_100": precision_at_k(probs, y_test, 100),
    })

results_df = pd.DataFrame(results).round(3)
print(results_df.to_string(index=False))

                model  roc_auc  avg_precision  precision_at_50  precision_at_100
Baseline (rule-based)    0.538          0.537             0.72              0.64
  Logistic Regression    0.594          0.591             0.68              0.67
        Decision Tree    0.611          0.580             0.52              0.53
        Random Forest    0.612          0.598             0.70              0.65


## 5. Limitations

*What this work cannot claim.*

In [10]:
limitations = [
    "Split-dependent result: re-running this exact model comparison on a "
    "different split flipped the ranking (baseline beat every trained "
    "model in an earlier run). The result in Section 4 reflects this split, "
    "not a split-independent truth.",

    "Proxy label, not a measured outcome: is_declining_label comes from a "
    "categorical trend field, not a directly observed future traffic drop.",

    "Observational, not causal: position, CTR, staleness, and word count "
    "are measured alongside decline, not manipulated. This cannot prove "
    "that refreshing a page causes traffic to recover.",

    "Does not predict or explain Google's ranking algorithm.",

    "Counterintuitive signal, unexplained: long-form content declined more "
    "than short content in this dataset. Reported as observed, not as a "
    "recommendation to shorten pages.",

    "Single-snapshot scope: the starter dataset is a fixed extract; "
    "seasonal effects and any change in FlyRank's flags since extraction "
    "are not captured.",

    "Decision-support only: every output here is meant for human review, "
    "not automated action.",
]

print("What this work cannot claim:")
for i, item in enumerate(limitations, 1):
    print(f"{i}. {item}")

What this work cannot claim:
1. Split-dependent result: re-running this exact model comparison on a different split flipped the ranking (baseline beat every trained model in an earlier run). The result in Section 4 reflects this split, not a split-independent truth.
2. Proxy label, not a measured outcome: is_declining_label comes from a categorical trend field, not a directly observed future traffic drop.
3. Observational, not causal: position, CTR, staleness, and word count are measured alongside decline, not manipulated. This cannot prove that refreshing a page causes traffic to recover.
4. Does not predict or explain Google's ranking algorithm.
5. Counterintuitive signal, unexplained: long-form content declined more than short content in this dataset. Reported as observed, not as a recommendation to shorten pages.
6. Single-snapshot scope: the starter dataset is a fixed extract; seasonal effects and any change in FlyRank's flags since extraction are not captured.
7. Decision-support

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [11]:
# Score the FULL catalog (not just the test split) using the strongest model
best_model = models["Random Forest"]  # winner from Section 4's honest table
df['decline_score'] = best_model.predict_proba(df[feature_cols])[:, 1]

def assign_action(row):
    if row['decline_score'] < 0.5:
        return "monitor", "Low decline risk — recheck next cycle"
    if row['is_striking'] == 1 and row['ctr_filled'] < df['ctr_filled'].median():
        return "refresh_and_review_ctr", "High impressions + striking distance + low CTR"
    if row['engagement_rate_filled'] < df['engagement_rate_filled'].quantile(0.25):
        return "refresh_and_review_engagement", "Low engagement relative to catalog"
    if row['days_since_last_update_filled'] > 180:
        return "expand_and_refresh", "Old content, high decline risk — deepen coverage"
    return "refresh", "Declining trend, standard update"

def confidence_band(score):
    if score >= 0.75 or score <= 0.25:
        return "high"
    if score >= 0.6 or score <= 0.4:
        return "medium"
    return "low"

df[['action', 'reason']] = df.apply(lambda r: pd.Series(assign_action(r)), axis=1)
df['confidence'] = df['decline_score'].apply(confidence_band)

# --- Action mix across the full catalog ---
print("Action mix:")
print((df['action'].value_counts(normalize=True) * 100).round(2))
print()
print("Confidence mix:")
print((df['confidence'].value_counts(normalize=True) * 100).round(2))
print()

# --- Top-ranked pages for this week's review ---
ranked = df.sort_values('decline_score', ascending=False).head(10)
ranked_display = ranked[['content_id', 'decline_score', 'action', 'reason']].copy()
ranked_display['rank'] = range(1, len(ranked_display) + 1)
ranked_display['score'] = (ranked_display['decline_score'] * 100).round(0).astype(int)
print(ranked_display[['rank', 'content_id', 'score', 'action', 'reason']].to_string(index=False))

Action mix:
action
refresh                   57.28
monitor                   31.66
refresh_and_review_ctr    10.85
expand_and_refresh         0.21
Name: proportion, dtype: float64

Confidence mix:
confidence
medium    41.54
low       30.50
high      27.95
Name: proportion, dtype: float64

 rank           content_id  score  action                           reason
    1 content_4bcb30ab7531     97 refresh Declining trend, standard update
    2 content_252c884e4400     96 refresh Declining trend, standard update
    3 content_ab82c4705992     96 refresh Declining trend, standard update
    4 content_267d8cf80905     96 refresh Declining trend, standard update
    5 content_a027a8c7ff1f     96 refresh Declining trend, standard update
    6 content_3e5673c16aa7     96 refresh Declining trend, standard update
    7 content_b018d113bdaf     96 refresh Declining trend, standard update
    8 content_b61233d4bc91     96 refresh Declining trend, standard update
    9 content_bb704201c499     96 r

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [12]:
import json

# --- 1. Model comparison table (Section 4 of the paper) ---
results_df.to_csv("artifacts_model_comparison.csv", index=False)

# --- 2. Signal audit bars (staleness / CTR / word count / flag test) ---
def decline_rate(mask):
    return round(df.loc[mask, 'is_declining_label'].mean() * 100, 1)

signal_audit = {
    "stale_91_180d":      decline_rate((df['days_since_last_update_filled'] >= 91) & (df['days_since_last_update_filled'] <= 180)),
    "fresh_0_30d":        decline_rate(df['days_since_last_update_filled'] <= 30),
    "low_ctr_striking":   decline_rate((df['is_striking'] == 1) & (df['ctr_filled'] < df['ctr_filled'].median())),
    "high_ctr_striking":  decline_rate((df['is_striking'] == 1) & (df['ctr_filled'] >= df['ctr_filled'].median())),
    "long_form_3500+":    decline_rate(df['word_count_filled'] >= 3500),
    "short_form_lt1000":  decline_rate(df['word_count_filled'] < 1000),
}
with open("artifacts_signal_audit.json", "w") as f:
    json.dump(signal_audit, f, indent=2)

# --- 3. Action mix + confidence mix (Section 6) ---
action_mix = (df['action'].value_counts(normalize=True) * 100).round(2).to_dict()
confidence_mix = (df['confidence'].value_counts(normalize=True) * 100).round(2).to_dict()
with open("artifacts_playbook_mix.json", "w") as f:
    json.dump({"action_mix": action_mix, "confidence_mix": confidence_mix}, f, indent=2)

print("Artifacts written:")
print("- artifacts_model_comparison.csv")
print("- artifacts_signal_audit.json")
print("- artifacts_playbook_mix.json")
print()
print("Signal audit:", signal_audit)
print("Action mix:", action_mix)
print("Confidence mix:", confidence_mix)

Artifacts written:
- artifacts_model_comparison.csv
- artifacts_signal_audit.json
- artifacts_playbook_mix.json

Signal audit: {'stale_91_180d': np.float64(61.1), 'fresh_0_30d': np.float64(51.1), 'low_ctr_striking': np.float64(59.6), 'high_ctr_striking': np.float64(55.4), 'long_form_3500+': np.float64(59.7), 'short_form_lt1000': np.float64(20.7)}
Action mix: {'refresh': 57.28, 'monitor': 31.66, 'refresh_and_review_ctr': 10.85, 'expand_and_refresh': 0.21}
Confidence mix: {'medium': 41.54, 'low': 30.5, 'high': 27.95}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
